# Large Retrieval Evaluation Notebook

This notebook is for building a larger Bacardi product database and comparing retrievers. It intentionally avoids page-parsing details and focuses on retrieval quality: ProductCards, LLM-generated embedding texts, FAISS/OpenAI embeddings, and BM25 over concise profile text.

Run from the repository root. The expensive/live steps are guarded by flags.

## 1. Environment And Run Flags

In [1]:
from pathlib import Path
import json
import math
import os
import sys
from collections import Counter
from pprint import pprint

import llm_module

ROOT = Path.cwd()
DATA = ROOT / "data"
RAW_DIR = DATA / "raw_pages"
PARSED_DIR = DATA / "parsed_products"
PRODUCTS_DIR = DATA / "catalog" / "products"
PROFILES_DIR = DATA / "catalog" / "search_profiles"
INDEX_DIR = DATA / "indexes"

print("cwd:", ROOT)
print("python:", sys.version.split()[0])
print("OPENAI_API_KEY configured:", bool(llm_module.OPENAI_API_KEY))
print("OPENAI_MODEL:", llm_module.OPENAI_MODEL)
print("OPENAI_EMBEDDING_MODEL:", llm_module.OPENAI_EMBEDDING_MODEL)


cwd: c:\Users\egorg\Documents\SPIRT_TEST
python: 3.12.12
OPENAI_API_KEY configured: True
OPENAI_MODEL: gpt-5.4-mini
OPENAI_EMBEDDING_MODEL: text-embedding-3-small


In [2]:
# Flip these intentionally. Full crawl / LLM parsing / real embeddings can cost time and tokens.
RUN_FULL_CRAWL = True
RUN_LLM_PARSE_ALL_PRODUCTS = True
RUN_LLM_SEARCHABLE_TEXT_BUILD = True
RUN_REAL_FAISS_INDEX_BUILD = True

# Use this while iterating prompts/costs. Set to None for full available dataset.
LIMIT_PRODUCTS = None

print("RUN_FULL_CRAWL:", RUN_FULL_CRAWL)
print("RUN_LLM_PARSE_ALL_PRODUCTS:", RUN_LLM_PARSE_ALL_PRODUCTS)
print("RUN_LLM_SEARCHABLE_TEXT_BUILD:", RUN_LLM_SEARCHABLE_TEXT_BUILD)
print("RUN_REAL_FAISS_INDEX_BUILD:", RUN_REAL_FAISS_INDEX_BUILD)
print("LIMIT_PRODUCTS:", LIMIT_PRODUCTS)


RUN_FULL_CRAWL: True
RUN_LLM_PARSE_ALL_PRODUCTS: True
RUN_LLM_SEARCHABLE_TEXT_BUILD: True
RUN_REAL_FAISS_INDEX_BUILD: True
LIMIT_PRODUCTS: None


## 2. Full Site Database Build Pipeline

These cells build the larger database. They do not inspect page parsing details; they only show artifact counts and errors.

In [3]:
from sommelier.ingestion.crawl_bacardi import crawl_bacardi

if not RUN_FULL_CRAWL:
    print("Skipping full crawl. Set RUN_FULL_CRAWL = True to refresh all Bacardi pages.")
else:
    crawl_summary = crawl_bacardi(max_products=None, delay_seconds=0.25)
    print(crawl_summary.model_dump_json(indent=2))


{
  "listing_url": "https://www.bacardi.com/our-rums/",
  "raw_html_files": [
    "data\\raw_pages\\www-bacardi-com-our-rums.html",
    "data\\raw_pages\\www-bacardi-com-our-rums-anejo-cuatro-rum.html",
    "data\\raw_pages\\www-bacardi-com-our-rums-bacardi-tropical.html",
    "data\\raw_pages\\www-bacardi-com-our-rums-carta-blanca-rum.html",
    "data\\raw_pages\\www-bacardi-com-our-rums-carta-negra-rum.html",
    "data\\raw_pages\\www-bacardi-com-our-rums-carta-oro-rum.html",
    "data\\raw_pages\\www-bacardi-com-our-rums-coconut-rum.html",
    "data\\raw_pages\\www-bacardi-com-our-rums-coquito.html",
    "data\\raw_pages\\www-bacardi-com-our-rums-dragonberry-rum.html",
    "data\\raw_pages\\www-bacardi-com-our-rums-lime-rum.html",
    "data\\raw_pages\\www-bacardi-com-our-rums-limon-rum.html",
    "data\\raw_pages\\www-bacardi-com-our-rums-raspberry-rum.html",
    "data\\raw_pages\\www-bacardi-com-our-rums-reserva-diez-rum.html",
    "data\\raw_pages\\www-bacardi-com-our-rums-reserv

In [4]:
from sommelier.ingestion.llm_product_parser import parse_directory

if not RUN_LLM_PARSE_ALL_PRODUCTS:
    print("Skipping LLM ProductCard parsing. Set RUN_LLM_PARSE_ALL_PRODUCTS = True to parse all page records.")
elif not llm_module.OPENAI_API_KEY:
    print("Skipping LLM ProductCard parsing: OPENAI_API_KEY is not configured.")
else:
    product_summary = parse_directory(
        input_dir=PARSED_DIR,
        output_dir=PRODUCTS_DIR,
        limit=LIMIT_PRODUCTS,
        force=True,
    )
    print(product_summary.model_dump_json(indent=2))


c:\Users\egorg\anaconda3\envs\myenv\Lib\site-packages\langchain_openai\chat_models\base.py:453: UserWarning: Invalid schema for OpenAI's structured output feature, which is the default method for `with_structured_output` as of langchain-openai==0.3. Specify `method="function_calling"` instead or update your schema. See supported schemas: https://platform.openai.com/docs/guides/structured-outputs#supported-schemas
  warnings.warn(message)


{
  "input_dir": "c:\\Users\\egorg\\Documents\\SPIRT_TEST\\data\\parsed_products",
  "output_dir": "c:\\Users\\egorg\\Documents\\SPIRT_TEST\\data\\catalog\\products",
  "results": [
    {
      "input_file": "c:\\Users\\egorg\\Documents\\SPIRT_TEST\\data\\parsed_products\\www-bacardi-com-our-rums-anejo-cuatro-rum.json",
      "output_file": "c:\\Users\\egorg\\Documents\\SPIRT_TEST\\data\\catalog\\products\\www-bacardi-com-our-rums-anejo-cuatro-rum.json",
      "product_id": "bacardi-anejo-cuatro-rum",
      "skipped": false,
      "dry_run": false,
      "warnings": [
        "Navigation, related-product lists, and footer content were present and ignored.",
        "FAQ section included general rum information; only supported answers were extracted.",
        "No explicit ABV, barrel type, country of origin, or ingredients beyond the FAQ text were present."
      ]
    },
    {
      "input_file": "c:\\Users\\egorg\\Documents\\SPIRT_TEST\\data\\parsed_products\\www-bacardi-com-our-rums

In [5]:
from sommelier.catalog.search_profiles import build_search_profiles, load_search_profiles, build_searchable_text_prompt

if not RUN_LLM_SEARCHABLE_TEXT_BUILD:
    print("Skipping LLM searchable_text build. Set RUN_LLM_SEARCHABLE_TEXT_BUILD = True to regenerate embedding texts.")
else:
    if not llm_module.OPENAI_API_KEY:
        print("Skipping LLM searchable_text build: OPENAI_API_KEY is not configured.")
    else:
        profile_paths = build_search_profiles(
            input_dir=PRODUCTS_DIR,
            output_dir=PROFILES_DIR,
            force=True,
            limit=LIMIT_PRODUCTS,
            use_llm_searchable_text=True,
        )
        print(f"Built {len(profile_paths)} ProductSearchProfile files with LLM searchable_text.")
        for path in profile_paths:
            print(" -", path)


Built 15 ProductSearchProfile files with LLM searchable_text.
 - c:\Users\egorg\Documents\SPIRT_TEST\data\catalog\search_profiles\www-bacardi-com-our-rums-anejo-cuatro-rum.json
 - c:\Users\egorg\Documents\SPIRT_TEST\data\catalog\search_profiles\www-bacardi-com-our-rums-bacardi-tropical.json
 - c:\Users\egorg\Documents\SPIRT_TEST\data\catalog\search_profiles\www-bacardi-com-our-rums-carta-blanca-rum.json
 - c:\Users\egorg\Documents\SPIRT_TEST\data\catalog\search_profiles\www-bacardi-com-our-rums-carta-negra-rum.json
 - c:\Users\egorg\Documents\SPIRT_TEST\data\catalog\search_profiles\www-bacardi-com-our-rums-carta-oro-rum.json
 - c:\Users\egorg\Documents\SPIRT_TEST\data\catalog\search_profiles\www-bacardi-com-our-rums-coconut-rum.json
 - c:\Users\egorg\Documents\SPIRT_TEST\data\catalog\search_profiles\www-bacardi-com-our-rums-coquito.json
 - c:\Users\egorg\Documents\SPIRT_TEST\data\catalog\search_profiles\www-bacardi-com-our-rums-dragonberry-rum.json
 - c:\Users\egorg\Documents\SPIRT_TES

In [6]:
from sommelier.retrieval.faiss_index import FaissIndex, OpenAIEmbeddingProvider, build_index_from_profiles

if not RUN_REAL_FAISS_INDEX_BUILD:
    print("Skipping real FAISS/OpenAI embedding index build. Set RUN_REAL_FAISS_INDEX_BUILD = True to rebuild it.")
elif not llm_module.OPENAI_API_KEY:
    print("Skipping index build: OPENAI_API_KEY is not configured.")
else:
    index = build_index_from_profiles(PROFILES_DIR, embedding_provider=OpenAIEmbeddingProvider())
    index.save(INDEX_DIR)
    print(f"Indexed {len(index.profiles)} profiles with {llm_module.OPENAI_EMBEDDING_MODEL}.")
    print("metadata:", INDEX_DIR / "product_faiss_metadata.json")
    print("native faiss exists:", (INDEX_DIR / "product.faiss").exists())


Indexed 15 profiles with text-embedding-3-small.
metadata: c:\Users\egorg\Documents\SPIRT_TEST\data\indexes\product_faiss_metadata.json
native faiss exists: True


## 3. Database Statistics And Validation

In [7]:
def count_files(path, pattern="*.json"):
    return len(list(path.glob(pattern))) if path.exists() else 0

stats = {
    "raw_html": count_files(RAW_DIR, "*.html"),
    "parsed_pages": count_files(PARSED_DIR, "*.json"),
    "product_cards": count_files(PRODUCTS_DIR, "*.json"),
    "search_profiles": count_files(PROFILES_DIR, "*.json"),
    "index_metadata_exists": (INDEX_DIR / "product_faiss_metadata.json").exists(),
    "native_faiss_exists": (INDEX_DIR / "product.faiss").exists(),
}
pprint(stats)


{'index_metadata_exists': True,
 'native_faiss_exists': True,
 'parsed_pages': 17,
 'product_cards': 15,
 'raw_html': 16,
 'search_profiles': 15}


In [8]:
profiles = load_search_profiles(PROFILES_DIR) if PROFILES_DIR.exists() else []
print("profiles loaded:", len(profiles))

for profile in profiles:
    problems = []
    if not profile.searchable_text.strip():
        problems.append("empty searchable_text")
    if len(profile.searchable_text) < 120:
        problems.append("very short searchable_text")
    if len(profile.searchable_text) > 1400:
        problems.append("very long searchable_text")
    if problems:
        print(profile.product_id, problems)


profiles loaded: 15


## 4. Inspect Embedding Texts

This is the key debugging view: these are the exact texts embedded into FAISS.

In [9]:
for profile in profiles:
    print("=" * 110)
    print(f"{profile.name} [{profile.product_id}]")
    print("category:", profile.category)
    print("flavor_tags:", profile.flavor_tags)
    print("usage_tags:", profile.usage_tags)
    print("cocktail_names:", profile.cocktail_names)
    print("\nEMBEDDING TEXT:")
    print(profile.searchable_text)
    print()


BACARDÍ Añejo Cuatro Rum [bacardi-anejo-cuatro-rum]
category: gold rum
flavor_tags: ['vanilla', 'oak', 'honey', 'clove', 'cinnamon', 'caramel']
usage_tags: ['cocktail', 'highball']
cocktail_names: ['Cuatro Highball', 'Cuatro Mismo', 'Cuatro Remix', 'Cuatro Smash', 'Cuatro Cuba Libre']

EMBEDDING TEXT:
BACARDÍ Añejo Cuatro Rum is a gold rum aged for at least four years, with a soft golden color and a smooth, refined character. It opens with aromas of vanilla and cinnamon, followed by a palate of dark honey and clove. The finish brings toffee and oak, with mild vanilla, toasted oak, and a smooth finish overall. Best served in cocktails or highballs, it adds sophistication to mixed drinks and suits rum drinks that highlight honey, caramel, vanilla, oak, cinnamon, and spice.

BACARDÍ Tropical [bacardi-tropical]
category: flavoured rum
flavor_tags: ['citrus', 'pineapple', 'coconut', 'guava']
usage_tags: ['cocktail', 'mixer']
cocktail_names: ['Tropical & Soda', 'Tropical & Lemonade', 'Tropic

## 5. Prompt Debugging For `searchable_text`

Use this section to inspect the current prompt and optionally regenerate one product's embedding text.

In [10]:
selected_product_path = sorted(PRODUCTS_DIR.glob("*.json"))[0] if PRODUCTS_DIR.exists() and list(PRODUCTS_DIR.glob("*.json")) else None
print("selected_product_path:", selected_product_path)

if selected_product_path:
    selected_card = json.loads(selected_product_path.read_text(encoding="utf-8"))
    selected_profile = next((p for p in profiles if Path(p.source_product_card_path).name == selected_product_path.name), None)
    if selected_profile:
        prompt_preview = build_searchable_text_prompt(
            card=selected_card,
            flavor_tags=selected_profile.flavor_tags,
            usage_tags=selected_profile.usage_tags,
            cocktail_names=selected_profile.cocktail_names,
            tasting_summary=selected_profile.tasting_summary or "",
        )
        print(prompt_preview[:5000])


selected_product_path: c:\Users\egorg\Documents\SPIRT_TEST\data\catalog\products\www-bacardi-com-our-rums-anejo-cuatro-rum.json
Create one compact natural-language retrieval text for embedding search.
The text must describe only the current rum product.
Make it maximally informative for semantic search, but avoid repetition.
Include product identity, style/category, flavor profile, aroma/palate/finish, texture or smoothness, serve style, cocktail suitability, and useful flavor/use tags in organic language.
Do not include FAQ, source metadata, legal text, warnings, or recommended related rums.
Do not invent facts. Prefer 70-130 words. Return plain text only, no JSON, no bullets.

PRODUCT DATA:
{
  "name": "BACARDÍ Añejo Cuatro Rum",
  "brand": "BACARDÍ",
  "category": "gold rum",
  "short_description": "A blend of rums aged for a minimum of four years under the Caribbean sun, BACARDÍ Añejo Cuatro 4 year old rum has a unique taste and a soft, golden colour.",
  "marketing_description": "

In [11]:
# Optional single-item regeneration for prompt tuning.
RUN_SINGLE_SEARCHABLE_TEXT_REGEN = False

if not RUN_SINGLE_SEARCHABLE_TEXT_REGEN:
    print("Skipping single searchable_text regeneration.")
elif not llm_module.OPENAI_API_KEY:
    print("Skipping: OPENAI_API_KEY is not configured.")
else:
    from sommelier.catalog.search_profiles import product_card_to_search_profile

    regenerated = product_card_to_search_profile(
        selected_card,
        selected_product_path,
        use_llm_searchable_text=True,
    )
    print(regenerated.searchable_text)


Skipping single searchable_text regeneration.


## 6. Retriever Implementations: BM25 And FAISS

In [12]:
from sommelier.retrieval.query_normalizer import normalize_query
from sommelier.retrieval.food_pairing_query import normalize_food_pairing_query, search_for_food_pairing

def tokenize(text):
    normalized = "".join(ch.lower() if ch.isalnum() else " " for ch in text)
    return [token for token in normalized.split() if len(token) > 1]

def bm25_search(query, profiles, top_k=5, k1=1.5, b=0.75):
    docs = [tokenize(profile.searchable_text) for profile in profiles]
    query_tokens = tokenize(normalize_query(query))
    avgdl = sum(len(doc) for doc in docs) / max(len(docs), 1)
    doc_freq = Counter()
    for doc in docs:
        for token in set(doc):
            doc_freq[token] += 1
    results = []
    total_docs = len(docs)
    for profile, doc in zip(profiles, docs):
        tf = Counter(doc)
        score = 0.0
        matched = []
        for token in query_tokens:
            if not tf[token]:
                continue
            matched.append(token)
            idf = math.log(1 + (total_docs - doc_freq[token] + 0.5) / (doc_freq[token] + 0.5))
            denom = tf[token] + k1 * (1 - b + b * len(doc) / max(avgdl, 1))
            score += idf * (tf[token] * (k1 + 1)) / denom
        results.append({"profile": profile, "score": score, "matched_tokens": sorted(set(matched))})
    return sorted(results, key=lambda item: item["score"], reverse=True)[:top_k]

def load_real_faiss_index():
    if not (INDEX_DIR / "product_faiss_metadata.json").exists():
        print("No index metadata found. Build the index first.")
        return None
    if not llm_module.OPENAI_API_KEY:
        print("OPENAI_API_KEY not configured; FAISS/OpenAI search skipped.")
        return None
    return FaissIndex.load(INDEX_DIR, embedding_provider=OpenAIEmbeddingProvider())

faiss_index = load_real_faiss_index()
print("faiss_index loaded:", faiss_index is not None)


faiss_index loaded: True


## 7. Query Set

In [13]:
direct_queries = [
    "I want a vanilla and oak rum for cocktails.",
    "smooth aged rum with honey, clove and toffee",
    "tropical fruity rum with pineapple coconut and guava",
    "light white rum for mojito or daiquiri",
    "rum for highball with oak and spice",
]

food_queries = [
    "шашлык из свинины",
    "стейк",
    "рыба",
    "десерт",
    "острое блюдо",
    "гречка с грибами",
]

print("direct queries:", len(direct_queries))
print("food queries:", len(food_queries))


direct queries: 5
food queries: 6


## 8. Direct Search: BM25 vs FAISS

In [14]:
def print_bm25_results(results, top_k=5):
    for rank, item in enumerate(results[:top_k], start=1):
        profile = item["profile"]
        print(f"{rank}. {profile.name} [{profile.product_id}] score={item['score']:.3f}")
        print("   matched_tokens:", item["matched_tokens"])
        print("   text:", profile.searchable_text[:500])

def print_faiss_results(results, top_k=5):
    for rank, result in enumerate(results[:top_k], start=1):
        profile = result.profile
        print(f"{rank}. {profile.name} [{profile.product_id}] score={result.score:.3f}")
        print("   flavor_tags:", profile.flavor_tags)
        print("   usage_tags:", profile.usage_tags)
        print("   text:", profile.searchable_text[:500])


In [15]:
for query in direct_queries:
    print("=" * 120)
    print("QUERY:", query)
    print("NORMALIZED:", normalize_query(query))

    print("\nBM25:")
    print_bm25_results(bm25_search(query, profiles, top_k=5))

    print("\nFAISS / OpenAI embeddings:")
    if faiss_index is None:
        print("skipped")
    else:
        print_faiss_results(faiss_index.search(query, top_k=5))
    print()


QUERY: I want a vanilla and oak rum for cocktails.
NORMALIZED: Rum with vanilla, oak flavors. Suitable for cocktails and mixing drinks.

BM25:
1. BACARDÍ Añejo Cuatro Rum [bacardi-anejo-cuatro-rum] score=3.994
   matched_tokens: ['and', 'cocktails', 'drinks', 'for', 'oak', 'rum', 'vanilla', 'with']
   text: BACARDÍ Añejo Cuatro Rum is a gold rum aged for at least four years, with a soft golden color and a smooth, refined character. It opens with aromas of vanilla and cinnamon, followed by a palate of dark honey and clove. The finish brings toffee and oak, with mild vanilla, toasted oak, and a smooth finish overall. Best served in cocktails or highballs, it adds sophistication to mixed drinks and suits rum drinks that highlight honey, caramel, vanilla, oak, cinnamon, and spice.
2. BACARDÍ Carta Blanca [bacardi-carta-blanca] score=3.978
   matched_tokens: ['and', 'cocktails', 'for', 'mixing', 'oak', 'rum', 'vanilla', 'with']
   text: BACARDÍ Carta Blanca is a classic white rum made for m

## 9. Food-Pairing Search: Query Expansion Then Retrieval

In [16]:
for food_query in food_queries:
    expanded = normalize_food_pairing_query(food_query)
    print("=" * 120)
    print("FOOD:", food_query)
    print("EXPANDED:", expanded)

    print("\nBM25 over expanded query:")
    print_bm25_results(bm25_search(expanded, profiles, top_k=5))

    print("\nFAISS over expanded query:")
    if faiss_index is None:
        print("skipped")
    else:
        pairing = search_for_food_pairing(food_query, top_k=5, index=faiss_index)
        print("CAVEAT:", pairing.caveat)
        print_faiss_results(pairing.retrieval_results)
    print()


FOOD: шашлык из свинины
EXPANDED: Rum suitable for grilled barbecue meat. Rich, smoky, spiced rum profile. Rum with caramel, spice, oak or molasses for pork.

BM25 over expanded query:
1. BACARDÍ Carta Oro [bacardi-carta-oro] score=2.857
   matched_tokens: ['caramel', 'oak', 'rum', 'with']
   text: BACARDÍ Carta Oro is a gold rum with a mellow, toasted character and golden color, aged in toasted oak barrels and charcoal filtered. It opens with a nutty, buttery aroma, then shows a rich, toasted palate of vanilla, buttery caramel, toasted almond, sweet banana, and zesty orange peel. The finish is light, oaky, dry, and slightly sweet, with a smooth, soothing texture. Best served in bold, punchy cocktails, especially a Cuba Libre or rum toddy. Useful search tags: gold rum, vanilla, caramel, ba
2. BACARDÍ Carta Negra [bacardi-carta-negra] score=2.788
   matched_tokens: ['caramel', 'oak', 'rum', 'with']
   text: BACARDÍ Carta Negra is a medium-bodied dark rum with rich black-rum character, e

## 10. Manual Judgement Notes

Use this space to record which retriever looks better per query. Do not treat this as an automated benchmark yet; the dataset is still small unless the full crawl and parse steps were run.

In [17]:
manual_notes = {
    "I want a vanilla and oak rum for cocktails.": "",
    "smooth aged rum with honey, clove and toffee": "",
    "tropical fruity rum with pineapple coconut and guava": "",
    "light white rum for mojito or daiquiri": "",
    "шашлык из свинины": "",
    "стейк": "",
    "рыба": "",
    "десерт": "",
}
pprint(manual_notes)


{'I want a vanilla and oak rum for cocktails.': '',
 'light white rum for mojito or daiquiri': '',
 'smooth aged rum with honey, clove and toffee': '',
 'tropical fruity rum with pineapple coconut and guava': '',
 'десерт': '',
 'рыба': '',
 'стейк': '',
 'шашлык из свинины': ''}


In [1]:
from sommelier.ingestion.crawl_bacardi_cocktails import crawl_bacardi_cocktails

crawl_summary = crawl_bacardi_cocktails(
    delay_seconds=0.5,
)

print(crawl_summary.model_dump_json(indent=2))
print("Discovered cocktail URLs:", len(crawl_summary.cocktail_urls))



{
  "listing_url": "https://www.bacardi.com/rum-cocktails/",
  "raw_html_files": [
    "data\\raw_cocktail_pages\\www-bacardi-com-rum-cocktails.html",
    "data\\raw_cocktail_pages\\www-bacardi-com-rum-cocktails-bacardi-and-cola.html",
    "data\\raw_cocktail_pages\\www-bacardi-com-rum-cocktails-coquito.html",
    "data\\raw_cocktail_pages\\www-bacardi-com-rum-cocktails-daiquiri.html",
    "data\\raw_cocktail_pages\\www-bacardi-com-rum-cocktails-mojito.html",
    "data\\raw_cocktail_pages\\www-bacardi-com-rum-cocktails-ocho-old-fashioned.html",
    "data\\raw_cocktail_pages\\www-bacardi-com-rum-cocktails-pina-colada.html",
    "data\\raw_cocktail_pages\\www-bacardi-com-rum-cocktails-zombie.html",
    "data\\raw_cocktail_pages\\www-bacardi-com-rum-cocktails-blood-moon.html",
    "data\\raw_cocktail_pages\\www-bacardi-com-rum-cocktails-frozen-pina-colada.html",
    "data\\raw_cocktail_pages\\www-bacardi-com-rum-cocktails-frozen-strawberry-daiquiri.html",
    "data\\raw_cocktail_pages\\ww

In [3]:
from pathlib import Path
from sommelier.ingestion.llm_cocktail_parser import parse_directory

parse_summary = parse_directory(
    input_dir=Path("data/parsed_cocktails"),
    output_dir=Path("data/catalog/cocktails"),
    force=True,
)

print(parse_summary.model_dump_json(indent=2))

{
  "input_dir": "data\\parsed_cocktails",
  "output_dir": "data\\catalog\\cocktails",
  "results": [
    {
      "input_file": "data\\parsed_cocktails\\www-bacardi-com-rum-cocktails-bacardi-and-cola.json",
      "output_file": "data\\catalog\\cocktails\\www-bacardi-com-rum-cocktails-bacardi-and-cola.json",
      "cocktail_id": "bacardi-and-cola",
      "skipped": false,
      "dry_run": false,
      "warnings": [
        "Recipe steps on the page appear duplicated/redundant; preserved as written and condensed into ordered steps.",
        "Main rum is explicit as BACARDÍ Carta Blanca rum, also referenced as BACARDÍ Superior in FAQ; used page's primary rum naming."
      ]
    },
    {
      "input_file": "data\\parsed_cocktails\\www-bacardi-com-rum-cocktails-blood-moon.json",
      "output_file": "data\\catalog\\cocktails\\www-bacardi-com-rum-cocktails-blood-moon.json",
      "cocktail_id": "blood-moon",
      "skipped": false,
      "dry_run": false,
      "warnings": []
    },
    {

In [4]:
import json
from pathlib import Path

cards = sorted(Path("data/catalog/cocktails").glob("*.json"))
print("Cocktail cards:", len(cards))

for path in cards[:2]:
    card = json.loads(path.read_text(encoding="utf-8"))
    print("\n---", path.name)
    print("name:", card["name"])
    print("main_rum:", card.get("main_rum"))
    print("short:", card.get("short_description"))
    print("marketing:", card.get("marketing_description"))
    print("ingredients:", card["recipe"]["ingredients"][:6])
    print("steps:", card["recipe"]["steps"][:3])
    print("warnings:", card.get("extraction_warnings"))

Cocktail cards: 42

--- www-bacardi-com-rum-cocktails-bacardi-and-cola.json
name: BACARDÍ & Cola
main_rum: BACARDÍ Carta Blanca rum
short: A classic Bacardi rum and cola made with BACARDÍ Carta Blanca and chilled cola over ice.
marketing: Like many of the world's greatest pairings, a rum and cola is best if made with 'the original' BACARDÍ. BACARDÍ Carta Blanca rum was designed to be the ultimate mixing spirit.
ingredients: [{'name': 'BACARDÍ Carta Blanca rum', 'amount': '50 ml'}, {'name': 'cola', 'amount': '100 ml'}]
steps: ['Fill a highball glass with ice.', 'Pour in the BACARDÍ Carta Blanca white rum, followed by chilled cola, and give it all a gentle stir.', 'Add the chilled cola.']
warnings: ['Recipe steps on the page appear duplicated/redundant; preserved as written and condensed into ordered steps.', "Main rum is explicit as BACARDÍ Carta Blanca rum, also referenced as BACARDÍ Superior in FAQ; used page's primary rum naming."]

--- www-bacardi-com-rum-cocktails-blood-moon.json
n

In [5]:
import json
from pathlib import Path
from collections import Counter

cards = []
for path in sorted(Path("data/catalog/cocktails").glob("*.json")):
    card = json.loads(path.read_text(encoding="utf-8"))
    cards.append((path, card))

print("Cocktail cards:", len(cards))
print("Missing main_rum:", sum(1 for _, c in cards if not c.get("main_rum")))
print("Missing ingredients:", sum(1 for _, c in cards if not c["recipe"].get("ingredients")))
print("Missing steps:", sum(1 for _, c in cards if not c["recipe"].get("steps")))

warnings = Counter()
for _, c in cards:
    for w in c.get("extraction_warnings", []):
        warnings[w] += 1

print("\nTop warnings:")
for warning, count in warnings.most_common(10):
    print(count, "-", warning)

Cocktail cards: 42
Missing main_rum: 0
Missing ingredients: 0
Missing steps: 0

Top warnings:
1 - Recipe steps on the page appear duplicated/redundant; preserved as written and condensed into ordered steps.
1 - Main rum is explicit as BACARDÍ Carta Blanca rum, also referenced as BACARDÍ Superior in FAQ; used page's primary rum naming.
1 - The page explicitly names BACARDÍ Gran Reserva Limitada rum in the instructions, while the ingredient line says BACARDÍ Reserva Limitada rum; kept both as shown and used the ingredient line as the recipe ingredient.
1 - No explicit prep time or difficulty provided.
1 - No explicit garnish beyond 'no garnish' provided.
